Creating Functions

In [2]:
from pathlib import Path
import os
#from infer_subc.organelles.membrane import membrane_composite, close_and_filter, masked_object_thresh_bind_pm, mix_nuc_and_fill, double_watershed
from infer_subc.core.img import make_aggregate, scale_and_smooth, masked_object_thresh, fill_and_filter_linear_size
from infer_subc.organelles.cellmask import non_linear_cellmask_transform
from infer_subc.organelles.nuclei import infer_nuclei_fromcytoplasm

from infer_subc.core.file_io import (list_image_files, 
                                     read_czi_image,
                                     export_inferred_organelle)
import napari
viewer = napari.Viewer()
import numpy as np

USER INPUT Loading image


Change the path for your raw imgages or deconvolved INPUT folder in_data_path in orange between quotes

change the path for your segmented objects OUTPUT folder out_data_path

test image change the number depending on your list

In [3]:
data_root_path = Path(os.path.expanduser("~")) / "Documents/Python Scripts/Infer-subc-2D"

in_data_path = data_root_path /"Z:/Cohen Lab/Maria Clara/2_Lab data/1_Multispectral data/2023/112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2/Deconvolved iN day28 images water Cp/tiff"
out_data_path = data_root_path/"Z:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmentation iNday28/InferSubC_iNday28/nuc edited"
im_type = ".tiff"

img_file_list = list_image_files(in_data_path,im_type)

In [4]:
test_img_n = 15

test_img_name = img_file_list[test_img_n]

if not Path.exists(out_data_path):
    Path.mkdir(out_data_path)
    print(f"making {out_data_path}")

img_data,meta_dict = read_czi_image(test_img_name)
channel_names = meta_dict['name']
img = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']
viewer.add_image(img_data, scale=scale)

<Image layer 'img_data' at 0x19b098e6a70>

Step 1 SELECT CHANNEL that in negative resamble the Nuclei and Smooth

In [5]:
agg = make_aggregate(img_in=img_data,
                     weight_ch0 = 0,
                     weight_ch1= 0,
                     weight_ch2= 1,
                     weight_ch3= 1,
                     weight_ch4= 1,
                     weight_ch5= 5,
                     weight_ch6= 20,
                     weight_ch7= 0,
                     weight_ch9= 0,
                     rescale= True)
viewer.add_image(agg, scale=scale)
sns = scale_and_smooth(img_in=agg, 
                       median_size= 0, 
                       gauss_sigma= 4, 
                       slice_by_slice= True)
viewer.add_image(sns, scale=scale)

<Image layer 'sns' at 0x19b08c66140>

SEGMENTING Nuclei

In [6]:
mot = masked_object_thresh(structure_img_smooth= sns, 
                           global_method='tri', 
                           cutoff_size=500, 
                           local_adjust=0.2)
viewer.add_image(mot, scale=scale)
fafls = fill_and_filter_linear_size(img=mot, 
                                    hole_min=0, 
                                    hole_max=0, 
                                    min_size=5, 
                                    method= "3D", 
                                    connectivity= 1)
viewer.add_image(fafls, scale=scale)
infc = infer_nuclei_fromcytoplasm(cytoplasm_mask=fafls, 
                                  nuc_min_width=20,
                                  nuc_max_width=100,
                                  fill_filter_method="3D",
                                  small_obj_width=20)
viewer.add_image(infc, scale=scale)

<Image layer 'infc' at 0x19b85a2bee0>

Export

In [7]:
export_inferred_organelle(infc, name="nuc", meta_dict=meta_dict, out_data_path=out_data_path)

saved file: 02102024_MSi08L_iN_Day28_BR1a_N17_Unmixing_0_cmle.ome-nuc
